# Multi-Class Cloud Cover Threshold Comparison: CART vs RF vs KNN vs SVM vs XGBoost

This notebook sweeps different CLOUDY_PIXEL_PERCENTAGE thresholds across all five
multi-class classifiers to find the optimal cloud cover threshold.

> Uses the same training points, bands, and classifier hyperparameters as the
> individual notebooks in `nb/multi/`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import ee
import geemap

# ── Constants ────────────────────────────────────────────────────────────────

CAMPUS_GEOJSON = {
    "type": "Polygon",
    "coordinates": [
        [
            [80.01710357666015, 23.173962177472703],
            [80.03259601593017, 23.165361215115187],
            [80.03654422760009, 23.172502420044232],
            [80.026802444458, 23.181694681000845],
            [80.01542987823485, 23.176960548201308],
        ]
    ],
}

BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI", "NDBI", "SAVI"]
SEED = 42

WATER_POINTS_ASSET = "users/cosypix/water_points"
FOREST_POINTS_ASSET = "users/cosypix/forest_points"
SOIL_POINTS_ASSET = "users/cosypix/soil_points"
BUILDINGS_POINTS_ASSET = "users/cosypix/buildings_points"

CLASSIFIER_CONFIGS = {
    "cart": {"maxNodes": 10, "minLeafPopulation": 10},
    "rf": {"numberOfTrees": 200, "minLeafPopulation": 1, "bagFraction": 0.3, "maxNodes": 10},
    "knn": {"k": 5, "searchMethod": "AUTO", "metric": "EUCLIDEAN"},
    "svm": {"kernelType": "RBF", "gamma": 1.0, "cost": 100.0},
    "xgb": {"numberOfTrees": 200, "shrinkage": 0.05, "maxNodes": 5},
}

CLASSIFIER_DISPLAY_NAMES = {
    "cart": "CART",
    "rf": "Random Forest",
    "knn": "k-NN",
    "svm": "SVM (RBF)",
    "xgb": "XGBoost",
}

def init_ee():
    """Initialize Earth Engine using EE_PROJECT_ID from the .env file."""
    load_dotenv(find_dotenv())
    ee_project = os.getenv("EE_PROJECT_ID")
    if not ee_project:
        raise ValueError("EE_PROJECT_ID not set in .env file")
    ee.Initialize(project=ee_project)
    print("Earth Engine initialized successfully.")

def get_campus_geometry():
    """Return the campus boundary as an ee.Geometry."""
    return ee.Geometry(CAMPUS_GEOJSON)

def mask_s2_clouds(image):
    """Apply Sentinel-2 QA60 cloud/cirrus mask and scale to reflectance."""
    qa = image.select("QA60")
    cloud = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

def load_training_points():
    """Load and merge multi-class training point FeatureCollections."""
    water = ee.FeatureCollection(WATER_POINTS_ASSET)
    forest = ee.FeatureCollection(FOREST_POINTS_ASSET)
    soil = ee.FeatureCollection(SOIL_POINTS_ASSET)
    buildings = ee.FeatureCollection(BUILDINGS_POINTS_ASSET)
    training_points = water.merge(forest).merge(soil).merge(buildings)
    return water, forest, soil, buildings, training_points

def sample_and_split(image, training_points, bands=None, seed=None, split=0.7):
    """Sample *image* at *training_points* and split into train/test sets."""
    if bands is None:
        bands = BANDS
    if seed is None:
        seed = SEED
    training = image.select(bands).sampleRegions(
        collection=training_points,
        properties=["label"],
        scale=10,
    )
    training = training.filter(ee.Filter.notNull(bands + ["label"]))
    training = training.randomColumn("random", seed)
    train_set = training.filter(ee.Filter.lt("random", split))
    test_set = training.filter(ee.Filter.gte("random", split))
    return train_set, test_set

def create_classifier(model_type):
    """Return an **untrained** ee.Classifier with canonical hyperparameters."""
    if model_type not in CLASSIFIER_CONFIGS:
        raise ValueError(f"Unknown model_type '{model_type}'. Choose from {list(CLASSIFIER_CONFIGS.keys())}")
    params = CLASSIFIER_CONFIGS[model_type]
    if model_type == "cart":
        return ee.Classifier.smileCart(**params)
    elif model_type == "rf":
        return ee.Classifier.smileRandomForest(**params)
    elif model_type == "knn":
        return ee.Classifier.smileKNN(**params)
    elif model_type == "svm":
        return ee.Classifier.libsvm(**params)
    elif model_type == "xgb":
        return ee.Classifier.smileGradientTreeBoost(**params)

def get_classifier_factories():
    return {
        CLASSIFIER_DISPLAY_NAMES[key]: (lambda k=key: create_classifier(k))
        for key in CLASSIFIER_CONFIGS
    }


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

init_ee()

### Campus Boundary

In [ ]:
campus = get_campus_geometry()

### Set Bands

In [ ]:
bands = BANDS

### Load Training Points

In [ ]:
water_points, forest_points, soil_points, buildings_points, training_points = load_training_points()

print(f"Water points:     {water_points.size().getInfo()}")
print(f"Forest points:    {forest_points.size().getInfo()}")
print(f"Soil points:      {soil_points.size().getInfo()}")
print(f"Buildings points: {buildings_points.size().getInfo()}")

### Define Classifiers & Cloud Cover Thresholds

In [ ]:
CLOUD_COVER_VALUES = [5, 10, 15, 20, 30, 50]

CLASSIFIERS = get_classifier_factories()

print(f"Classifiers: {list(CLASSIFIERS.keys())}")
print(f"Cloud Cover Thresholds: {CLOUD_COVER_VALUES}")

### Run Cloud Cover Sweep

In [ ]:
results = []

for cc in CLOUD_COVER_VALUES:
    print(f"{'='*60}")
    print(f"  Cloud Cover Threshold: {cc}%")
    print(f"{'='*60}")

    # Build the image for this cloud cover threshold
    dataset = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterDate('2026-01-01', '2026-02-28')
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cc))
               .map(mask_s2_clouds))

    image_count = dataset.size().getInfo()
    print(f"  Available images: {image_count}")

    if image_count == 0:
        print(f"  ⚠ No images found for {cc}% threshold – skipping")
        for clf_name in CLASSIFIERS:
            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': 0,
            })
        continue

    image = dataset.median().clip(campus)

    # Add spectral indices
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    savi = image.expression(
        '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
        {'NIR': image.select('B8'), 'RED': image.select('B4')}
    ).rename('SAVI')
    image = image.addBands([ndvi, ndwi, ndbi, savi])

    # Sample and split 
    train_set, test_set = sample_and_split(image, training_points, bands)

    for clf_name, clf_factory in CLASSIFIERS.items():
        try:
            classifier = clf_factory().train(
                features=train_set,
                classProperty='label',
                inputProperties=bands
            )

            validated = test_set.classify(classifier)
            cm = validated.errorMatrix('label', 'classification')
            accuracy = cm.accuracy().getInfo()
            kappa = cm.kappa().getInfo()

            print(f"  {clf_name:30s}  Accuracy: {accuracy:.4f}   Kappa: {kappa:.4f}")

            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': accuracy, 'kappa': kappa, 'image_count': image_count,
            })
        except Exception as e:
            print(f"  {clf_name:30s}  ERROR: {e}")
            results.append({
                'cloud_cover': cc, 'classifier': clf_name,
                'accuracy': None, 'kappa': None, 'image_count': image_count,
            })

print("✅ Sweep complete!")

### Results Table

In [ ]:
df = pd.DataFrame(results)
print(df[['cloud_cover', 'classifier', 'accuracy', 'kappa', 'image_count']].to_string(index=False))

### Accuracy & Kappa vs Cloud Cover

In [ ]:
colors = {
    "CART": "#9b59b6",
    "Random Forest": "#2ecc71",
    "k-NN": "#f39c12",
    "SVM (RBF)": "#e74c3c",
    "XGBoost": "#3498db",
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy plot
for clf_name in CLASSIFIERS:
    subset = df[df['classifier'] == clf_name].dropna(subset=['accuracy'])
    ax1.plot(subset['cloud_cover'], subset['accuracy'], 'o-', 
             label=clf_name, color=colors.get(clf_name, '#95a5a6'), linewidth=2, markersize=6)

ax1.set_xlabel('Cloud Cover Threshold (%)', fontweight='bold')
ax1.set_ylabel('Accuracy', fontweight='bold')
ax1.set_title('Multi-Class Accuracy vs Cloud Cover Threshold', fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_xticks(CLOUD_COVER_VALUES)

# Kappa plot
for clf_name in CLASSIFIERS:
    subset = df[df['classifier'] == clf_name].dropna(subset=['kappa'])
    ax2.plot(subset['cloud_cover'], subset['kappa'], 's--', 
             label=clf_name, color=colors.get(clf_name, '#95a5a6'), linewidth=2, markersize=6)

ax2.set_xlabel('Cloud Cover Threshold (%)', fontweight='bold')
ax2.set_ylabel('Kappa', fontweight='bold')
ax2.set_title('Multi-Class Kappa vs Cloud Cover Threshold', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xticks(CLOUD_COVER_VALUES)

plt.tight_layout()
plt.savefig('multi_cloud_cover_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fe/multi_cloud_cover_comparison.png")

### Save Results

In [ ]:
df.to_csv('multi_cloud_cover_results.csv', index=False)
print("Saved: fe/multi_cloud_cover_results.csv")

# Summary
print("" + "="*60)
print("  BEST ACCURACY PER CLOUD COVER THRESHOLD")
print("="*60)
for cc in CLOUD_COVER_VALUES:
    subset = df[(df['cloud_cover'] == cc) & df['accuracy'].notna()]
    if not subset.empty:
        best = subset.loc[subset['accuracy'].idxmax()]
        print(f"  Cloud Cover {cc:3d}%  {best['classifier']:20s}  Accuracy: {best['accuracy']:.4f}")

print("\n" + "="*60)
print("  BEST OVERALL ACCURACY PER CLASSIFIER")
print("="*60)
for clf_name in CLASSIFIERS:
    subset = df[(df['classifier'] == clf_name) & df['accuracy'].notna()]
    if not subset.empty:
        best = subset.loc[subset['accuracy'].idxmax()]
        print(f"  {clf_name:20s}  Cloud Cover: {int(best['cloud_cover']):3d}%  Accuracy: {best['accuracy']:.4f}")